# CUTEst

In [ ]:
import multiprocessing
import os
from typing import Tuple, List, TypeAlias, Any

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from zipfile import BadZipFile

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.problem.cutest import CUTEstQNProblem
from qnlab.solver.qn import qn
from qnlab.util.callback import Callback
from qnlab.util.method import Method, get_methods
from qnlab.experiment.profile import performance_profile
from qnlab.experiment.vis import vis

In [ ]:
task_type: TypeAlias = Tuple[str, Method, dict, int, np.float64]


def get_file_path(task: task_type) -> str:
    prob_name, method, _option, precision, noise = task
    prob_type = "noisy" if noise > 0 else str(precision)

    return f"../data/temp/{prob_type}/{prob_name}/{method.to_label()}.npz"


def solveProblemWithTimeout(task: task_type) -> str:
    prob_name, method, option, precision, noise = task
    try:
        prob = CUTEstQNProblem(prob_name, precision=precision, noise=noise)
        callback = Callback()
        qn(prob, method, option, callback)
        file_path = get_file_path(task)
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        np.savez_compressed(
            file_path,
            calls=callback.calls,
            fxs=callback.fxs,
            gnorms=callback.gnorms,
        )
        return f"✓ {prob_name} with {method.to_label()}"
    except Exception as e:
        return f"✗ {prob_name} with {method.to_label()}: {str(e)}"


def run(
    problems: list[str],
    methods: list[Tuple[Method, dict]],
    delete_mode: str,
    precision: int,
    noise: np.float64,
    TL: int,
):
    # Prepare all tasks
    tasks: List[task_type] = []
    for problem in problems:
        for method, option in methods:
            task = (problem, method, option, precision, noise)
            npz_path = get_file_path(task)
            if os.path.exists(npz_path):
                if delete_mode == "None":
                    continue
                elif delete_mode == "Hamaguchi" and not method.base.startswith(
                    "Hamaguchi"
                ):
                    continue
            tasks.append(task)
    print(f"Total tasks to run: {len(tasks)}")

    errors = []  # 後でまとめて報告するためのエラー記録リスト
    with multiprocessing.Pool(processes=1) as pool:
        for i, task in enumerate(tasks):
            file_path = get_file_path(task)
            try:
                result = pool.apply_async(solveProblemWithTimeout, (task,))
                msg = result.get(timeout=TL)
                print(f"[{i + 1}/{len(tasks)}] ✅ Success: {msg}")
            except multiprocessing.TimeoutError:
                print(f"[{i + 1}/{len(tasks)}] ⏱ Timeout: {file_path}")
                pool.terminate()
                pool.join()
                pool = multiprocessing.Pool(processes=1)  # restart the pool
                errors.append((file_path, "Timeout"))
            except Exception as e:
                print(f"[{i + 1}/{len(tasks)}] ⚠ Error: {file_path}: {e}")
                errors.append((file_path, f"Error: {e}"))

    # 全タスク終了後、エラーがあればまとめて例外を投げる
    if errors:
        msg = "\n".join([f" - {name}: {err}" for name, err in errors])
        print("=" * 20)

        print(f"The following tasks failed:\n{msg}")
        print("=" * 20)

In [ ]:
def individual_plot(
    problems: list[str],
    methods: list[Tuple[Method, dict]],
    precision: int,
    noise: np.float64,
):
    # Perform visualization using loaded data
    for prob_name in problems:
        prob = CUTEstQNProblem(prob_name, precision=precision)

        callbacks = []
        for method, option in methods:
            task: task_type = (prob_name, method, option, precision, noise)
            file_path = get_file_path(task)

            assert os.path.exists(file_path), file_path

            data = np.load(file_path)
            calls = data["calls"]
            fxs = data["fxs"]
            gnorms = data["gnorms"]
            callback = Callback()
            callback.calls = calls.astype(int)
            callback.fxs = fxs
            callback.gnorms = gnorms
            callback.xs = [np.zeros(0) for _ in range(len(calls))]
            callbacks.append(callback)

        labels = [method.to_label() for method, _ in methods]
        vis(prob, callbacks, labels, prob_name, only_grad=True, only_plot=True)


def make_df(
    methods: list[Tuple[Method, dict]],
    problems: list[str],
    precision: int,
    noise: np.float64,
    gtol: float,
) -> Tuple[list[str], np.ndarray, Any]:
    alg_names = list(method.to_label() for method, _ in methods)
    nAlgorithms = len(alg_names)
    nProbs = len(problems)
    callsM = np.zeros((nAlgorithms, nProbs), dtype=float)
    fxsM = np.zeros((nAlgorithms, nProbs), dtype=float)
    gnormsM = np.zeros((nAlgorithms, nProbs), dtype=float)

    for j, prob_name in enumerate(problems):
        for i, (method, option) in enumerate(methods):
            task: task_type = (prob_name, method, option, precision, noise)
            file_path = get_file_path(task)
            # assert os.path.exists(file_path), file_path

            if not os.path.exists(file_path):
                print(f"Warning: Missing file {file_path}")
                res = (np.inf, np.inf, np.inf)
            else:
                try:
                    data = np.load(file_path)
                except EOFError:
                    print(file_path)
                    raise EOFError
                except BadZipFile:
                    print(file_path)
                    raise
                calls = data["calls"]
                fxs = data["fxs"]
                gnorms = data["gnorms"]
                if len(calls) == 0:
                    res = (np.inf, np.inf, np.inf)
                else:
                    isOk = gnorms <= gtol
                    if not np.any(isOk):
                        res = (np.inf, np.inf, np.inf)
                    else:
                        idx = np.where(isOk)[0][0]
                        assert calls[idx] >= 0
                        # We take max with 1
                        # Otherwise, performance profile will cause error
                        call_max_1 = max(1, calls[idx])
                        res = (call_max_1, fxs[idx], gnorms[idx])

            callsM[i, j], fxsM[i, j], gnormsM[i, j] = res

    zero_rows = np.where(np.all(callsM == 0, axis=0))[0]
    callsM = np.delete(callsM, zero_rows, axis=1)
    gnormsM = np.delete(gnormsM, zero_rows, axis=1)
    problems = [problems[i] for i in range(len(problems)) if i not in zero_rows]

    # データフレームの作成
    data = {}
    data["problem"] = problems
    for i, alg_name in enumerate(alg_names):
        data[f"{alg_name}"] = callsM[i, :].tolist()
    df = pd.DataFrame(data)
    df.set_index("problem", inplace=True)

    min_calls = df.min(axis=1)
    print(min_calls)
    for alg_name in alg_names:
        num_of_first = (df[alg_name] == min_calls).sum()
        print(f"{alg_name}: {num_of_first} problems solved with minimum calls")
        num_of_solved = np.isfinite(df[alg_name]).sum()
        print(f"{alg_name}: {num_of_solved} problems solved successfully")

    our = "Hamaguchi"
    scipy = "SciPy"
    if our in df and scipy in df:
        our_wins = (df[our] < df[scipy]).sum()
        scipy_wins = (df[scipy] < df[our]).sum()
        ties = (df[our] == df[scipy]).sum()
        print(f"{our} wins: {our_wins}, {scipy} wins: {scipy_wins}, ties: {ties}")

    def color_scale_with_cmap(row):
        if np.all(np.isinf(row.values)):
            return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
        norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)  # type:ignore
        cmap = matplotlib.colormaps["coolwarm"]
        return [
            f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
            for r, g, b, _ in cmap(norm(row.values))
        ]

    styled_df = df.style.apply(color_scale_with_cmap, axis=1)

    return alg_names, callsM, styled_df

In [ ]:
import seaborn as sns
from pathlib import Path


tab20 = plt.colormaps.get_cmap("tab20")

color_palette = {
    "Hamaguchi": tab20(0),  # Red-ish
    "Hamaguchi-MS": tab20(1),  # Darker red variant
    "Line": tab20(2),  # Blue
    "Line-MS": tab20(3),  # Lighter blue
    "SciPy": tab20(6),  # Aqua/teal-like
    "Reg": tab20(4),  # Green
    "NTQN": tab20(8),  # Purple
}

line_styles = {
    "Hamaguchi": "o-",
    "Hamaguchi-MS": "o--",
    "Line": "^--",
    "Line-MS": "^-.",
    "SciPy": "v:",
    "Reg": "D--",
    "NTQN": "s-.",
}


def draw_pp(
    alg_names: list[str],
    callsM: np.ndarray,
    precision: int,
    noise: np.float64,
    gtol: float,
):
    # Set style for publication quality
    sns.set_style("whitegrid")
    plt.rcParams.update(
        {
            "text.usetex": True,
            "font.family": "serif",
            "font.size": 20,
            "figure.dpi": 300,
            "lines.linewidth": 2.0,
        }
    )

    # Assign colors and line styles based on method names
    colors = [color_palette.get(name, "black") for name in alg_names]
    line_styles_list = [line_styles.get(name, "o-") for name in alg_names]

    # Create figure with proper size for paper
    # Wider figure to accommodate legend on the right
    fig, ax = plt.subplots(figsize=(7, 5.5))

    # Draw performance profiles
    performance_profile(
        callsM.T,
        linestyle=line_styles_list,
        colors=colors,
        thetaMax=10.0,
        markersize=6,
        markevery=[0],
        linewidth=2.2,
    )

    # Customize the plot
    ax = plt.gca()
    # Legend removed from individual plots - will be added separately in LaTeX
    ax.set_xlabel(r"Performance Ratio $\tau$", fontsize=18, fontweight="normal")
    ax.set_ylabel(
        r"Proportion of Problems Solved $\rho_s(\tau)$",
        fontsize=18,
        fontweight="normal",
    )

    # Format gtol for display
    gtol_str = f"{gtol:.0e}"

    # Add precision/noise info to title
    if noise == 0:
        title_str = rf"precision={precision}, gtol={gtol_str}"
    else:
        title_str = rf"noise={noise:.0e}, gtol={gtol_str}"

    ax.set_title(title_str, fontsize=25, fontweight="normal", pad=15)

    # Grid styling
    ax.grid(True, alpha=0.35, linestyle="-", linewidth=0.6, color="gray")
    ax.set_axisbelow(True)

    # Improve spine visibility
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1.0)

    plt.tight_layout()

    # Save figure
    output_path = Path("/home/hirok/University/qnlab/doc_private/imgs/compare")
    output_path.mkdir(parents=True, exist_ok=True)

    precision_noise = f"precision{precision}" if noise == 0 else f"noise{noise}"
    gtol_filename = f"{gtol:.0e}".replace("+", "")
    pdf_path = output_path / f"_pp_{precision_noise}_gtol{gtol_filename}.pdf"
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight", dpi=300)
    print(f"Saved figure to {pdf_path}")

    plt.show()
    plt.close()

In [ ]:
TOO_LONG_TIME_PROBLEMS = [
    "DMN15103LS",
    "DMN15332LS",
    "DMN37142LS",
    "DMN37143LS",
    "EIGENALS",
    "EIGENBLS",
    "EIGENCLS",
]


def main():
    # Delete mode: "All" (delete all files), "Hamaguchi" (delete only method.base == "Hamaguchi"), "None" (delete nothing)
    delete_mode = "None"
    # delete_mode = input("delete mode? (All/Hamaguchi/None): ")
    # if delete_mode == "All":
    #     yesno = input("delete all temp files? (yes/no): ")
    #     if yesno.lower() != "yes":
    #         raise ValueError("Exiting without deleting temp files.")
    # elif delete_mode == "Hamaguchi":
    #     yesno = input("delete Hamaguchi temp files? (yes/no): ")
    #     if yesno.lower() != "yes":
    #         raise ValueError("Exiting without deleting temp files.")

    for precision, noise in [
        # (64, np.float64(0.0)),
        # (32, np.float64(0.0)),
        (16, np.float64(0.0)),
        (64, np.float64(1e-3)),
    ]:
        problems = problemsToRun(precision)
        problems = [p for p in problems if p not in TOO_LONG_TIME_PROBLEMS]
        methods = get_methods()
        if noise > 0:
            new_methods = []
            for method, option in methods:
                new_option = option.copy()
                new_option["gtol"] = noise * 10
                new_methods.append((method, new_option))
            methods = new_methods
        run(problems, methods, delete_mode, precision, noise, TL=300)

    methods = get_methods()
    for precision, noise, gtol in [
        (64, np.float64(0.0), 1e-3),
        (64, np.float64(0.0), 1e-4),
        (64, np.float64(0.0), 1e-5),
        (32, np.float64(0.0), 1e-3),
        (32, np.float64(0.0), 1e-4),
        (32, np.float64(0.0), 1e-5),
        (16, np.float64(0.0), 1e-3),
        (16, np.float64(0.0), 1e-4),
        (16, np.float64(0.0), 1e-5),
        (64, np.float64(1e-3), 1e-2),
    ]:
        problems = problemsToRun(precision)
        problems = [p for p in problems if p not in TOO_LONG_TIME_PROBLEMS]
        # individual_plot(problems, methods, precision, noise)
        alg_names, callsM, styled_df = make_df(
            methods, problems, precision, noise, gtol=gtol
        )
        display(styled_df)
        draw_pp(alg_names, callsM, precision, noise, gtol)


In [ ]:
main()

In [ ]:
def create_legend_only(alg_names: list[str]):
    """Create a legend-only figure for use in LaTeX layout"""
    import seaborn as sns
    from pathlib import Path

    sns.set_style("whitegrid")
    plt.rcParams.update(
        {
            "text.usetex": True,
            "font.family": "serif",
            "font.size": 20,
            "figure.dpi": 300,
            "lines.linewidth": 2.0,
        }
    )

    tab20 = plt.colormaps.get_cmap("tab20")

    color_palette = {
        "Hamaguchi": tab20(0),
        "Hamaguchi-MS": tab20(1),
        "Line": tab20(2),
        "Line-MS": tab20(3),
        "SciPy": tab20(6),
        "Reg": tab20(4),
        "NTQN": tab20(8),
    }

    line_styles = {
        "Hamaguchi": "o-",
        "Hamaguchi-MS": "o--",
        "Line": "^--",
        "Line-MS": "^-.",
        "SciPy": "v:",
        "Reg": "D--",
        "NTQN": "s-.",
    }

    # Create dummy lines for legend
    fig, ax = plt.subplots(figsize=(12, 1))
    handles = []
    for name in alg_names:
        color = color_palette.get(name, "black")
        linestyle = line_styles.get(name, "o-")
        (handle,) = plt.step(
            [0, 1],
            [0, 1],
            linestyle,
            color=color,
            linewidth=2.5,
            markersize=8,
        )
        handles.append(handle)

    # Create labels for legend
    legend_names = [name.replace("Hamaguchi", "Our") for name in alg_names]

    # Create legend with horizontal layout
    _legend = ax.legend(
        handles,
        legend_names,
        loc="center",
        framealpha=0.98,
        edgecolor="black",
        fancybox=True,
        fontsize=16,
        ncol=len(alg_names),
        frameon=True,
    )

    # Remove axes
    ax.axis("off")

    plt.tight_layout()

    # Save figure
    output_path = Path("/home/hirok/University/qnlab/doc_private/imgs/compare")
    output_path.mkdir(parents=True, exist_ok=True)

    legend_path = output_path / "_legend.pdf"
    fig.savefig(legend_path, format="pdf", bbox_inches="tight", dpi=300)
    print(f"Saved legend to {legend_path}")

    plt.close()


methods = get_methods()
alg_names = list(method.to_label() for method, _ in methods)
create_legend_only(alg_names)